# Stage 4 - adversarial attacks (FGSM / PGD, transfer to both models)

Wrap the CNN in ART, craft FGSM and PGD examples in the scaled space, and feed the SAME crafted frames to both the CNN and the Random Forest (a fair transfer test). Then cross-validate per-class robustness across the diversity gradient.

In [1]:
import sys
from pathlib import Path
try:
    import adversec
except ModuleNotFoundError:
    sys.path.insert(0, str(Path.cwd().parent)); import adversec
import numpy as np, pandas as pd
from adversec import config
from adversec.contract import FEATURES, LABEL_COLUMN, ID_COLUMN, DATA_COLUMNS
pd.set_option('display.width', 140)

# The two datasets are treated identically: every step below runs the SAME
# code for both. The only dataset-specific code in the project is each
# dataset's loader (adversec/datasets/ciciov.py, road.py).
DATASETS = ['ciciov2024', 'road']

## Setup - retrain the CNN + RF (ART needs the live model for gradients)

In [2]:
import torch, joblib
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from adversec.models import CNN1D, train_cnn, build_random_forest
from adversec.experiments import attack as atk
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', DEVICE)
def mf1(y, p): return f1_score(y, p, average='macro', zero_division=0)

device: cuda


## Epsilon sweep - macro-F1 under FGSM and PGD (both models)
The same crafted frames hit both models. Watch where each collapses.

In [3]:
wrapped = {}
for name in DATASETS:
    a = np.load(config.PROCESSED_DIR / f'{name}_stage2_arrays.npz')
    Xtr, ytr, Xte, yte = a['X_train'], a['y_train'], a['X_test'].astype(np.float32), a['y_test']
    classes = list(joblib.load(config.PROCESSED_DIR / f'{name}_label_encoder.joblib').classes_)
    cfg = config.load_dataset_config(name)
    cw = None
    if cfg.get('cnn_class_weights'):
        w = compute_class_weight('balanced', classes=np.unique(ytr), y=ytr)
        cw = torch.tensor(w, dtype=torch.float32, device=DEVICE)
    cnn = train_cnn(CNN1D(n_features=Xtr.shape[1], n_classes=len(classes)), Xtr, ytr, n_epochs=50, device=DEVICE, class_weights=cw)
    rf = build_random_forest().fit(Xtr, ytr)
    clf = atk.wrap_cnn_for_art(cnn, n_features=Xtr.shape[1], n_classes=len(classes), device=DEVICE)
    wrapped[name] = (clf, rf, Xte, yte, classes)
    print(f'\n=== {name} ===')
    print(f"{'eps':>6}{'FGSM_CNN':>10}{'FGSM_RF':>9}{'PGD_CNN':>9}{'PGD_RF':>9}")
    cc, cr = mf1(yte, clf.predict(Xte).argmax(1)), mf1(yte, rf.predict(Xte))
    print(f"{'clean':>6}{cc:>10.3f}{cr:>9.3f}{cc:>9.3f}{cr:>9.3f}")
    for eps in config.FGSM_EPSILONS:
        Xf = atk.generate_fgsm(clf, Xte, eps); Xp = atk.generate_pgd(clf, Xte, eps)
        print(f'{eps:>6.2f}{mf1(yte, clf.predict(Xf).argmax(1)):>10.3f}{mf1(yte, rf.predict(Xf)):>9.3f}'
              f'{mf1(yte, clf.predict(Xp).argmax(1)):>9.3f}{mf1(yte, rf.predict(Xp)):>9.3f}')

    epoch   1/50     loss 1.5561
    epoch   5/50     loss 0.1548
    epoch  10/50     loss 0.0145
    epoch  15/50     loss 0.0040
    epoch  20/50     loss 0.0024
    epoch  25/50     loss 0.0016
    epoch  30/50     loss 0.0012
    epoch  35/50     loss 0.0009
    epoch  40/50     loss 0.0009
    epoch  45/50     loss 0.0007
    epoch  50/50     loss 0.0007

=== ciciov2024 ===
   eps  FGSM_CNN  FGSM_RF  PGD_CNN   PGD_RF
 clean     0.675    0.776    0.675    0.776


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

  0.01     0.675    0.509    0.675    0.509


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

  0.05     0.372    0.149    0.412    0.150


...

  0.10     0.485    0.139    0.296    0.139


PGD - Batches:   0%|          | 0/250 [00:00<?, ?it/s]

  0.20     0.125    0.138    0.109    0.138


PGD - Batches:   0%|          | 0/250 [00:00<?, ?it/s]

  0.30     0.118    0.137    0.095    0.137


## Cross-validated per-class robustness vs the diversity gradient
Slower (5-fold, retrains per fold). To find out whether robustness follow signature count

In [4]:
from adversec.experiments.crossval import crossval_perclass_robustness
cv_results, cv_classes, cv_sig = {}, {}, {}
for name in DATASETS:
    strict = pd.read_csv(config.PROCESSED_DIR / f'{name}_strict.csv')
    classes = list(joblib.load(config.PROCESSED_DIR / f'{name}_label_encoder.joblib').classes_)
    print(f'\n=== {name}: 5-fold per-class F1 under PGD ===')
    res = crossval_perclass_robustness(strict, FEATURES, classes, config.FGSM_EPSILONS,
                                       attack='pgd', n_splits=5, device=DEVICE)
    sig = strict[LABEL_COLUMN].value_counts()
    cv_results[name], cv_classes[name], cv_sig[name] = res, classes, sig
    rows = []
    for i, c in enumerate(classes):
        row = {'class': c, 'signatures': int(sig.get(c, 0))}
        for e in ['clean'] + config.FGSM_EPSILONS:
            row[str(e)] = round(float(np.mean(res[e][i])), 3)
        rows.append(row)
    display(pd.DataFrame(rows).sort_values('signatures', ascending=False))


=== ciciov2024: 5-fold per-class F1 under PGD ===
    epoch   1/50     loss 1.7164


/home/koala/envs/adversec/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


    epoch   5/50     loss 0.2065
    epoch  10/50     loss 0.0162
    epoch  15/50     loss 0.0049
    epoch  20/50     loss 0.0031
    epoch  25/50     loss 0.0019
    epoch  30/50     loss 0.0017
    epoch  35/50     loss 0.0013
    epoch  40/50     loss 0.0011
    epoch  45/50     loss 0.0010
    epoch  50/50     loss 0.0009


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

...

PGD - Batches:   0%|          | 0/250 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/250 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/250 [00:00<?, ?it/s]

  fold 5/5 done


,class,signatures,clean,0.01,0.05,0.1,0.2,0.3
0,benign,21188,0.997,0.938,0.770,0.673,0.603,0.575
2,max-speedometer,10559,1.000,0.999,0.733,0.229,0.000,0.000
4,reverse-light-on,5994,0.997,0.895,0.027,0.000,0.000,0.000
3,reverse-light-off,1525,0.968,0.057,0.002,0.002,0.002,0.002
1,fuzzing,592,0.998,0.998,0.998,0.979,0.259,0.000


## Save adversarial results (for report writing + notebook 06)
Writes/updates `results/<name>_adversarial_results.json`: the cross-validated per-class PGD table and the distance-to-benign mechanism measurement. Merges rather than overwrites, so a `threat_sizing` section notebook 06 already saved here is preserved.

In [5]:
import json
from adversec.experiments.adversarial import distance_to_benign

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for name in DATASETS:
    res, classes, sig = cv_results[name], cv_classes[name], cv_sig[name]

    table = {}
    eps_keys = ['clean'] + list(config.FGSM_EPSILONS)
    for i, cname in enumerate(classes):
        table[cname] = {'signatures': int(sig.get(cname, 0)), 'by_epsilon': {}}
        for e in eps_keys:
            arr = np.array(res[e][i])
            table[cname]['by_epsilon'][str(e)] = {
                'mean': float(arr.mean()), 'std': float(arr.std()), 'folds': [float(x) for x in arr],
            }

    cfg = config.load_dataset_config(name)
    dist = distance_to_benign(name, res, classes, sig, benign_label=cfg['benign_label'])

    path = config.RESULTS_DIR / f'{name}_adversarial_results.json'
    existing = json.loads(path.read_text()) if path.exists() else {}
    existing.update({
        'dataset': name,
        'device': DEVICE,
        'attack_config': {
            'pgd_epsilon_sweep': list(config.FGSM_EPSILONS),
            'pgd_step_size': config.PGD_STEP_SIZE,
            'pgd_max_iter': config.PGD_MAX_ITER,
            'n_folds': 5,
            'seed': config.RANDOM_SEED,
        },
        'cross_validated_per_class_pgd': table,
        'mechanism_distance_to_benign': dist,
    })
    path.write_text(json.dumps(existing, indent=2))
    print('saved ->', path)

saved -> /home/koala/lab/adversec/results/ciciov2024_adversarial_results.json
saved -> /home/koala/lab/adversec/results/road_adversarial_results.json


## Ablation - does the distance-to-benign mechanism inversion survive without duplication?

CICIoV2024's `mechanism_distance_to_benign.pearson_r` is **negative** (classes farther from
benign are *less* robust), the opposite sign to ROAD's positive correlation. Before treating
that as a genuine data-structure finding, we need to rule out a confound: CICIoV2024's
scarcest classes (as few as 1 train signature) are padded up to 200 rows by repeating real
signatures (`duplicate_train_classes`, the "light-duplication convergence crutch"). This
re-runs the exact same 5-fold per-class PGD CV with that crutch switched off
(`use_duplication=False`), training on the real unpadded signatures only, and recomputes the
distance-to-benign correlation for comparison.

ROAD is not included here: `duplicate_train_classes` never fires for it in the first place
(its smallest per-fold class, fuzzing, has ~470 signatures, well above the 200 target), so
there is nothing to ablate. Expect some CICIoV2024 classes to train poorly or erratically
without the crutch (e.g. spoofing-GAS has only ~1 real training signature per fold) — that
instability is itself part of the answer, not a bug.

In [6]:
print('=== ciciov2024: ablation -- same 5-fold per-class PGD CV, WITHOUT the light-duplication convergence crutch ===')
strict_cic = pd.read_csv(config.PROCESSED_DIR / 'ciciov2024_strict.csv')
classes_cic = cv_classes['ciciov2024']
sig_cic = cv_sig['ciciov2024']

res_no_dup = crossval_perclass_robustness(
    strict_cic, FEATURES, classes_cic, config.FGSM_EPSILONS,
    attack='pgd', n_splits=5, device=DEVICE, use_duplication=False,
)

rows = []
for i, c in enumerate(classes_cic):
    row = {'class': c, 'signatures': int(sig_cic.get(c, 0))}
    for e in ['clean'] + config.FGSM_EPSILONS:
        row[str(e)] = round(float(np.mean(res_no_dup[e][i])), 3)
    rows.append(row)
display(pd.DataFrame(rows).sort_values('signatures', ascending=False))

cfg_cic = config.load_dataset_config('ciciov2024')
dist_no_dup = distance_to_benign('ciciov2024', res_no_dup, classes_cic, sig_cic, benign_label=cfg_cic['benign_label'])

dist_with_dup = json.loads((config.RESULTS_DIR / 'ciciov2024_adversarial_results.json').read_text())['mechanism_distance_to_benign']
print(f"\npearson_r WITH duplication (main result)      : {dist_with_dup['pearson_r']}")
print(f"pearson_r WITHOUT duplication (this ablation) : {dist_no_dup['pearson_r']}")

=== ciciov2024: ablation -- same 5-fold per-class PGD CV, WITHOUT the light-duplication convergence crutch ===
    epoch   1/50     loss 1.4774


/home/koala/envs/adversec/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


    epoch   5/50     loss 0.9776
    epoch  10/50     loss 0.8503
    epoch  15/50     loss 0.5608
    epoch  20/50     loss 0.4037
    epoch  25/50     loss 0.2890
    epoch  30/50     loss 0.1238
    epoch  35/50     loss 0.1272
    epoch  40/50     loss 0.0843
    epoch  45/50     loss 0.0391
    epoch  50/50     loss 0.0261


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

...

PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

  fold 5/5 done


,class,signatures,clean,0.01,0.05,0.1,0.2,0.3
1,benign,3547,0.998,0.998,0.917,0.848,0.335,0.032
0,DoS,21,0.956,0.938,0.314,0.000,0.000,0.000
3,spoofing-RPM,10,0.549,0.582,0.219,0.077,0.001,0.000
4,spoofing-SPEED,5,0.667,0.467,0.000,0.000,0.000,0.000
5,spoofing-STEERING_WHEEL,3,0.333,0.333,0.000,0.000,0.000,0.000
2,spoofing-GAS,2,0.400,0.400,0.000,0.000,0.000,0.000



pearson_r WITH duplication (main result)      : -0.59
pearson_r WITHOUT duplication (this ablation) : -0.295


## Save the ablation result

Writes `results/ciciov2024_ablation_no_duplication.json` — a separate file from
`ciciov2024_adversarial_results.json` so this doesn't touch the main result, just sits
alongside it for direct comparison.

In [7]:
ablation = {
    'dataset': 'ciciov2024',
    'purpose': (
        'Tests whether the distance-to-benign / robustness correlation inversion seen on '
        'CICIoV2024 (pearson_r negative, opposite sign to ROAD) survives without the '
        'light-duplication convergence crutch (train classes below 200 signatures are '
        'padded by repeating real rows), or is an artifact of it.'
    ),
    'with_duplication': dist_with_dup,
    'without_duplication': dist_no_dup,
}
path = config.RESULTS_DIR / 'ciciov2024_ablation_no_duplication.json'
path.write_text(json.dumps(ablation, indent=2))
print('saved ->', path)

saved -> /home/koala/lab/adversec/results/ciciov2024_ablation_no_duplication.json
